# Blind Screening Mitigation using SBERT

In this notebook, we added a simple mitigation step to the resume-job matching project. The goal of blind screening is to reduce the influence of demographic signals by removing identity-related information from resumes before calculating the matching score.

In the earlier notebooks, the model compared full resume variants that included fields such as name, pronouns, and university. Here, I remove those fields and create a blind-screened version of each resume. Then I use Sentence-BERT again to calculate resume-job similarity scores.

This allows me to compare the original SBERT results with blind-screened results and check whether removing demographic cues reduces score differences across counterfactual resume versions.

In [1]:
# Install the required libraries
!pip install sentence-transformers pandas scikit-learn

In [2]:
# Import libraries for data handling, embeddings, and similarity calculation
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
# Create folders for data and results
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [3]:
# Upload the required dataset files
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
    os.rename(filename, f"data/{filename}")
print("Files uploaded and moved to the data folder.")

Saving jobs.csv to jobs.csv
Saving resume_variants.csv to resume_variants.csv
Files uploaded and moved to the data folder.


In [4]:
# Load job descriptions and resume variants
jobs = pd.read_csv("data/jobs.csv")
resumes = pd.read_csv("data/resume_variants.csv")
print("Jobs shape:", jobs.shape)
print("Resume variants shape:", resumes.shape)
display(jobs.head())
display(resumes.head())

Jobs shape: (5, 5)
Resume variants shape: (40, 8)


,job_id,domain,title,company_name,job_description
0,J1,Software Engineering,Software Engineer,GOYT,Job Description:GOYT is seeking a skilled and ...
1,J2,Finance,Financial Analyst,Aya Healthcare,Aya Healthcare has an exciting 26-week contrac...
2,J3,Marketing,Marketing Coordinator,Corcoran Sawyer Smith,Job descriptionA leading real estate firm in N...
3,J4,Healthcare,Clinical Data Analyst,Insight Global,Must Have: 2+ years of current experience in a...
4,J5,Education,Academic Advisor,Kennesaw State University,About Us Are you ready to join a community lea...


,resume_id,domain,version,changed_signal,name,pronouns,university,resume_text
0,R1,Software Engineering,original,none,Emily Johnson,she/her,Stanford University,"Emily Johnson, she/her. B.S. in Computer Scien..."
1,R1,Software Engineering,name_changed,name,Aisha Khan,she/her,Stanford University,"Aisha Khan, she/her. B.S. in Computer Science ..."
2,R1,Software Engineering,pronoun_changed,pronoun,Emily Johnson,he/him,Stanford University,"Emily Johnson, he/him. B.S. in Computer Scienc..."
3,R1,Software Engineering,university_changed,university,Emily Johnson,she/her,Morgan State University,"Emily Johnson, she/her. B.S. in Computer Scien..."
4,R2,Software Engineering,original,none,Michael Smith,he/him,Massachusetts Institute of Technology,"Michael Smith, he/him. B.S. in Software Engine..."


In [6]:
# Create blind-screened resume text by removing identity-related words from resume_text
# This keeps qualification-related details while removing demographic cues
def create_blind_resume_text(row):
    text = str(row["resume_text"])
    # Remove name, pronouns, and university from the full resume text
    text = text.replace(str(row["name"]), "")
    text = text.replace(str(row["pronouns"]), "")
    text = text.replace(str(row["university"]), "")
    # Cleaning extra spaces
    text = " ".join(text.split())
    return text
resumes["blind_resume_text"] = resumes.apply(create_blind_resume_text, axis=1)
display(resumes[["resume_id", "version", "changed_signal", "blind_resume_text"]].head())

,resume_id,version,changed_signal,blind_resume_text
0,R1,original,none,", . B.S. in Computer Science from , 2024. Skil..."
1,R1,name_changed,name,", . B.S. in Computer Science from , 2024. Skil..."
2,R1,pronoun_changed,pronoun,", . B.S. in Computer Science from , 2024. Skil..."
3,R1,university_changed,university,", . B.S. in Computer Science from , 2024. Skil..."
4,R2,original,none,", . B.S. in Software Engineering from , 2023. ..."


In [7]:
# Combine job information into one text field for SBERT comparison
def create_job_text(row):
    return (
        str(row["title"]) + " " +
        str(row["domain"]) + " " +
        str(row["company_name"]) + " " +
        str(row["job_description"])
    )
jobs["job_text"] = jobs.apply(create_job_text, axis=1)
display(jobs[["job_id", "title", "job_text"]].head())

,job_id,title,job_text
0,J1,Software Engineer,Software Engineer Software Engineering GOYT Jo...
1,J2,Financial Analyst,Financial Analyst Finance Aya Healthcare Aya H...
2,J3,Marketing Coordinator,Marketing Coordinator Marketing Corcoran Sawye...
3,J4,Clinical Data Analyst,Clinical Data Analyst Healthcare Insight Globa...
4,J5,Academic Advisor,Academic Advisor Education Kennesaw State Univ...


In [8]:
# Load the Sentence-BERT model for blind-screened matching
model = SentenceTransformer("all-MiniLM-L6-v2")
print("SBERT model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT model loaded successfully.


In [9]:
# Compare every blind-screened resume with every job description
blind_results = []
for _, resume_row in resumes.iterrows():
    resume_embedding = model.encode([resume_row["blind_resume_text"]])
    for _, job_row in jobs.iterrows():
        job_embedding = model.encode([job_row["job_text"]])
        score = cosine_similarity(resume_embedding, job_embedding)[0][0]
        blind_results.append({
            "resume_id": resume_row["resume_id"],
            "version": resume_row["version"],
            "changed_signal": resume_row["changed_signal"],
            "job_id": job_row["job_id"],
            "job_title": job_row["title"],
            "blind_similarity_score": score
        })
blind_scores_df = pd.DataFrame(blind_results)
display(blind_scores_df.head())
print("Total blind-screened resume-job pairs scored:", len(blind_scores_df))

,resume_id,version,changed_signal,job_id,job_title,blind_similarity_score
0,R1,original,none,J1,Software Engineer,0.527946
1,R1,original,none,J2,Financial Analyst,0.197870
2,R1,original,none,J3,Marketing Coordinator,0.319795
3,R1,original,none,J4,Clinical Data Analyst,0.364824
4,R1,original,none,J5,Academic Advisor,0.323958


Total blind-screened resume-job pairs scored: 200


In [11]:
# Separate original and changed blind-screened scores
blind_original_scores = blind_scores_df[blind_scores_df["version"] == "original"].copy()
blind_changed_scores = blind_scores_df[blind_scores_df["version"] != "original"].copy()

blind_original_scores = blind_original_scores[[
    "resume_id",
    "job_id",
    "job_title",
    "blind_similarity_score"]]
blind_original_scores = blind_original_scores.rename(columns={
    "blind_similarity_score": "blind_original_score"})
blind_changed_scores = blind_changed_scores.rename(columns={
    "blind_similarity_score": "blind_changed_score"})
# Compare changed blind-screened scores with original blind-screened scores
blind_comparison = blind_changed_scores.merge(
    blind_original_scores,
    on=["resume_id", "job_id", "job_title"],
    how="left")
blind_comparison["blind_score_difference"] = (blind_comparison["blind_changed_score"] - blind_comparison["blind_original_score"])
blind_comparison["blind_absolute_difference"] = (blind_comparison["blind_score_difference"].abs())
display(blind_comparison.head())
print("Blind comparison rows:", blind_comparison.shape)

,resume_id,version,changed_signal,job_id,job_title,blind_changed_score,blind_original_score,blind_score_difference,blind_absolute_difference
0,R1,name_changed,name,J1,Software Engineer,0.527946,0.527946,0.0,0.0
1,R1,name_changed,name,J2,Financial Analyst,0.197870,0.197870,0.0,0.0
2,R1,name_changed,name,J3,Marketing Coordinator,0.319795,0.319795,0.0,0.0
3,R1,name_changed,name,J4,Clinical Data Analyst,0.364824,0.364824,0.0,0.0
4,R1,name_changed,name,J5,Academic Advisor,0.323958,0.323958,0.0,0.0


Blind comparison rows: (150, 9)


In [12]:
# Summarize score differences after blind screening
blind_summary = blind_comparison.groupby("changed_signal").agg(
    average_blind_score_difference=("blind_score_difference", "mean"),
    average_blind_absolute_difference=("blind_absolute_difference", "mean"),
    max_blind_absolute_difference=("blind_absolute_difference", "max"),
    min_blind_score_difference=("blind_score_difference", "min"),
    max_blind_score_difference=("blind_score_difference", "max")).reset_index()
display(blind_summary)

,changed_signal,average_blind_score_difference,average_blind_absolute_difference,max_blind_absolute_difference,min_blind_score_difference,max_blind_score_difference
0,name,0.000000,0.000000,0.000000,0.00000,0.000000
1,pronoun,0.000402,0.003617,0.012292,-0.00851,0.012292
2,university,0.000000,0.000000,0.000000,0.00000,0.000000


## Blind Screening Interpretation

The blind-screening results show that removing identity-related information reduced the counterfactual score differences. After removing name, pronouns, and university from the resume text, the original and modified resume versions produced the same or nearly the same SBERT similarity scores.

Compared with the original SBERT fairness results, the average absolute differences became much smaller after blind screening. Name and university differences were reduced to zero in this experiment, while pronoun-related differences were also reduced.

This suggests that blind screening is an effective mitigation step for this dataset because it prevents demographic cues from strongly influencing the resume-job matching score. However, this method also has limitations because removing information such as university may also remove context that could be considered relevant in some hiring settings.

In [13]:
# Save blind-screening results
blind_scores_df.to_csv("results/blind_sbert_scores.csv", index=False)
blind_comparison.to_csv("results/blind_fairness_comparison.csv", index=False)
blind_summary.to_csv("results/blind_fairness_summary.csv", index=False)
print("Blind-screened SBERT scores saved.")
print("Blind-screening fairness comparison saved.")
print("Blind-screening fairness summary saved.")

Blind-screened SBERT scores saved.
Blind-screening fairness comparison saved.
Blind-screening fairness summary saved.


In [14]:
# Compare every blind-screened resume with every job description
blind_results = []
for _, resume_row in resumes.iterrows():
    resume_embedding = model.encode([resume_row["blind_resume_text"]])
    for _, job_row in jobs.iterrows():
        job_embedding = model.encode([job_row["job_text"]])
        score = cosine_similarity(resume_embedding, job_embedding)[0][0]
        blind_results.append({
            "resume_id": resume_row["resume_id"],
            "version": resume_row["version"],
            "changed_signal": resume_row["changed_signal"],
            "job_id": job_row["job_id"],
            "job_title": job_row["title"],
            "blind_similarity_score": score})
blind_scores_df = pd.DataFrame(blind_results)
display(blind_scores_df.head())
print("Total blind-screened resume-job pairs scored:", len(blind_scores_df))

,resume_id,version,changed_signal,job_id,job_title,blind_similarity_score
0,R1,original,none,J1,Software Engineer,0.527946
1,R1,original,none,J2,Financial Analyst,0.197870
2,R1,original,none,J3,Marketing Coordinator,0.319795
3,R1,original,none,J4,Clinical Data Analyst,0.364824
4,R1,original,none,J5,Academic Advisor,0.323958


Total blind-screened resume-job pairs scored: 200


In [15]:
# Separate original blind-screened scores from changed blind-screened scores
blind_original_scores = blind_scores_df[blind_scores_df["version"] == "original"].copy()
blind_changed_scores = blind_scores_df[blind_scores_df["version"] != "original"].copy()
# Keep only needed columns from original scores
blind_original_scores = blind_original_scores[[
    "resume_id",
    "job_id",
    "job_title",
    "blind_similarity_score"]]
blind_original_scores = blind_original_scores.rename(columns={
    "blind_similarity_score": "blind_original_score"})
blind_changed_scores = blind_changed_scores.rename(columns={
    "blind_similarity_score": "blind_changed_score"})
# Match each changed blind-screened score with its original blind-screened score
blind_comparison = blind_changed_scores.merge(
    blind_original_scores,
    on=["resume_id", "job_id", "job_title"],
    how="left")
# Calculate score differences after blind screening
blind_comparison["blind_score_difference"] = (
    blind_comparison["blind_changed_score"] - blind_comparison["blind_original_score"])
blind_comparison["blind_absolute_difference"] = blind_comparison["blind_score_difference"].abs()
display(blind_comparison.head())
print("Blind comparison rows:", blind_comparison.shape)

,resume_id,version,changed_signal,job_id,job_title,blind_changed_score,blind_original_score,blind_score_difference,blind_absolute_difference
0,R1,name_changed,name,J1,Software Engineer,0.527946,0.527946,0.0,0.0
1,R1,name_changed,name,J2,Financial Analyst,0.197870,0.197870,0.0,0.0
2,R1,name_changed,name,J3,Marketing Coordinator,0.319795,0.319795,0.0,0.0
3,R1,name_changed,name,J4,Clinical Data Analyst,0.364824,0.364824,0.0,0.0
4,R1,name_changed,name,J5,Academic Advisor,0.323958,0.323958,0.0,0.0


Blind comparison rows: (150, 9)


In [16]:
# Summarize score differences after blind screening
blind_fairness_summary = blind_comparison.groupby("changed_signal").agg(
    average_blind_score_difference=("blind_score_difference", "mean"),
    average_blind_absolute_difference=("blind_absolute_difference", "mean"),
    max_blind_absolute_difference=("blind_absolute_difference", "max"),
    min_blind_score_difference=("blind_score_difference", "min"),
    max_blind_score_difference=("blind_score_difference", "max")).reset_index()
display(blind_fairness_summary)

,changed_signal,average_blind_score_difference,average_blind_absolute_difference,max_blind_absolute_difference,min_blind_score_difference,max_blind_score_difference
0,name,0.000000,0.000000,0.000000,0.00000,0.000000
1,pronoun,0.000402,0.003617,0.012292,-0.00851,0.012292
2,university,0.000000,0.000000,0.000000,0.00000,0.000000


## Blind Screening Result Interpretation

After applying blind screening, the score differences caused by name and university changes became zero. This shows that removing identity-related information from resumes can reduce the model’s sensitivity to demographic cues.

Pronoun-based differences were also very small after blind screening. Overall, this result suggests that blind screening is an effective mitigation step for this project because it reduces counterfactual score differences while still allowing SBERT to compare qualification-related resume content with job descriptions.

In [17]:
# Save blind-screened scores and fairness results
blind_scores_df.to_csv("results/blind_screening_scores.csv", index=False)
blind_comparison.to_csv("results/blind_screening_comparison.csv", index=False)
blind_fairness_summary.to_csv("results/blind_screening_summary.csv", index=False)
print("Blind-screening scores saved.")
print("Blind-screening comparison saved.")
print("Blind-screening fairness summary saved.")

Blind-screening scores saved.
Blind-screening comparison saved.
Blind-screening fairness summary saved.


In [18]:
# Download blind-screening result files for GitHub upload
from google.colab import files
files.download("results/blind_screening_scores.csv")
files.download("results/blind_screening_comparison.csv")
files.download("results/blind_screening_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>